# Task 1 - LoRA / QLoRA fine-tuning (Qwen2.5-0.5B-Instruct)
**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> **Run all**.

Total time on a T4: roughly 15-25 minutes. The last cell downloads `task1_results.zip` - send that file back.

## 1. GPU check and install

In [ ]:
!nvidia-smi
!pip install -q transformers peft accelerate datasets bitsandbytes
!pip uninstall -y -q torchao  # Colab ships torchao 0.10, which the new peft rejects

## 2. Write the scripts (identical to the ones in the task1_lora folder)

In [ ]:
%%writefile gen_data.py
"""Generate a synthetic 'plain English -> fixed JSON' dataset.
Train uses only the 'seen' templates. Test mixes seen templates with HELD-OUT templates,
unseen names and unseen tasks, so the fine-tuned model has realistic ways to fail.
Usage: python gen_data.py   (writes train.jsonl, test.jsonl)"""
import json, random

NAMES = ["Aarav", "Priya", "Rohan", "Meera", "Karan", "Sneha", "Vikram", "Ananya", "Dev", "Isha"]
EXTRA_NAMES = ["Tanvi", "Harsh", "Naman"]          # test only
TASKS = ["call the plumber", "submit the assignment", "buy groceries", "pay the electricity bill",
         "water the plants", "book train tickets", "renew my passport", "email the report",
         "go to the gym", "prepare for the viva"]
EXTRA_TASKS = ["pick up the parcel", "back up my laptop"]   # test only
WHENS = ["tomorrow at 9 am", "on Friday evening", "next Monday at 10:30", "tonight at 8",
         "in two hours", "on 25 December", "this Sunday morning", "at 6 pm today",
         "next week", "on the 3rd at noon"]
ITEMS = [("coffee", "food"), ("auto ride", "transport"), ("movie ticket", "entertainment"),
         ("notebook", "stationery"), ("pizza", "food"), ("metro card recharge", "transport"),
         ("headphones", "shopping"), ("doctor visit", "health"), ("phone recharge", "bills"),
         ("textbook", "education")]
MSGS = ["I will be late", "the meeting is moved to 4", "happy birthday", "please send the notes",
        "call me when free", "I reached home", "lunch at 1?", "the files are uploaded"]

T = {
 "set_reminder": {
  "seen": ["Remind me to {task} {when}", "Set a reminder: {task} {when}",
           "Can you remind me to {task} {when}?", "{when}, remind me to {task}"],
  "heldout": ["Don't let me forget to {task} {when}", "I need a nudge to {task} {when}, please"]},
 "add_expense": {
  "seen": ["I spent {amt} rupees on {item}", "Add an expense of {amt} for {item}", "Log {amt} rs {item}"],
  "heldout": ["Paid ₹{amt} for {item} just now", "{item} cost me {amt} bucks"]},
 "send_message": {
  "seen": ["Text {name} that {msg}", "Send a message to {name}: {msg}", "Tell {name} {msg}"],
  "heldout": ["Drop {name} a note saying {msg}", "WhatsApp {name} - {msg}"]},
}

def sample(rng, heldout, test):
    intent = rng.choice(list(T))
    t = rng.choice(T[intent]["heldout" if heldout else "seen"])
    if intent == "set_reminder":
        task = rng.choice(TASKS + (EXTRA_TASKS if test else [])); when = rng.choice(WHENS)
        text = t.format(task=task, when=when)
        obj = {"intent": intent, "task": task, "when": when}
    elif intent == "add_expense":
        item, cat = rng.choice(ITEMS)
        amt = rng.randint(20, 5000) if rng.random() < .6 else round(rng.uniform(10, 999), 1)
        text = t.format(amt=amt, item=item)
        obj = {"intent": intent, "amount": amt, "category": cat, "item": item}
    else:
        name = rng.choice(NAMES + (EXTRA_NAMES if test else [])); msg = rng.choice(MSGS)
        text = t.format(name=name, msg=msg)
        obj = {"intent": intent, "recipient": name, "message": msg}
    if rng.random() < .15: text = text.lower()
    return {"text": text, "target": json.dumps(obj, ensure_ascii=False), "heldout_template": heldout}

def build(n, heldout_frac, test, rng, exclude=()):
    seen, out = set(exclude), []
    while len(out) < n:
        ex = sample(rng, rng.random() < heldout_frac, test)
        if ex["text"] in seen: continue
        seen.add(ex["text"]); out.append(ex)
    return out

if __name__ == "__main__":
    rng = random.Random(7)
    train = build(1200, 0.0, False, rng)
    test = build(100, 0.5, True, rng, exclude=[e["text"] for e in train])
    for name, rows in (("train", train), ("test", test)):
        with open(f"{name}.jsonl", "w", encoding="utf-8") as f:
            for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(len(train), "train /", len(test), "test examples written")


In [ ]:
%%writefile train_lora.py
"""LoRA / QLoRA fine-tuning. Prompt tokens are masked so loss is only on the JSON answer.
Examples:
  python train_lora.py --model Qwen/Qwen2.5-0.5B-Instruct --out lora_r16
  python train_lora.py --model Qwen/Qwen2.5-1.5B-Instruct --qlora --out qlora_r16"""
import json, argparse, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments,
                          DataCollatorForSeq2Seq, BitsAndBytesConfig)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

PROMPT = "### Request:\n{text}\n### JSON:\n"
ap = argparse.ArgumentParser()
ap.add_argument("--model", default="Qwen/Qwen2.5-0.5B-Instruct")
ap.add_argument("--out", default="lora_out")
ap.add_argument("--r", type=int, default=16)
ap.add_argument("--alpha", type=int, default=32)
ap.add_argument("--dropout", type=float, default=0.05)
ap.add_argument("--lr", type=float, default=2e-4)
ap.add_argument("--epochs", type=float, default=3)
ap.add_argument("--bs", type=int, default=8)
ap.add_argument("--grad_acc", type=int, default=2)
ap.add_argument("--qlora", action="store_true")
a = ap.parse_args()

tok = AutoTokenizer.from_pretrained(a.model)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def encode(row):
    p = tok(PROMPT.format(text=row["text"]), add_special_tokens=False)["input_ids"]
    t = tok(row["target"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    return {"input_ids": p + t, "attention_mask": [1] * (len(p) + len(t)), "labels": [-100] * len(p) + t}

rows = [json.loads(l) for l in open("train.jsonl", encoding="utf-8")]
ds = Dataset.from_list(rows).map(encode, remove_columns=["text", "target", "heldout_template"])

if a.qlora:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(a.model, quantization_config=bnb, device_map="auto")
    model = prepare_model_for_kbit_training(model)
else:
    model = AutoModelForCausalLM.from_pretrained(a.model, torch_dtype=torch.float32).cuda()

cfg = LoraConfig(r=a.r, lora_alpha=a.alpha, lora_dropout=a.dropout, task_type="CAUSAL_LM",
                 target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, cfg)
model.print_trainable_parameters()

args = TrainingArguments(output_dir=a.out, per_device_train_batch_size=a.bs,
    gradient_accumulation_steps=a.grad_acc, learning_rate=a.lr, num_train_epochs=a.epochs,
    lr_scheduler_type="cosine", warmup_steps=7,  # 7 = 3% of 225 steps (warmup_ratio was removed in transformers 5)
    logging_steps=20, save_strategy="no",
    fp16=torch.cuda.is_available(), report_to="none",
    optim="paged_adamw_8bit" if a.qlora else "adamw_torch")
Trainer(model=model, args=args, train_dataset=ds,
        data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100)).train()
model.save_pretrained(a.out); tok.save_pretrained(a.out)
print("peak GPU memory (GB):", torch.cuda.max_memory_allocated() / 1e9)


In [ ]:
%%writefile eval_model.py
"""Evaluate on test.jsonl. Modes: zeroshot (base), fewshot (base + 3 examples), lora (adapter).
  python eval_model.py --mode zeroshot
  python eval_model.py --mode fewshot
  python eval_model.py --mode lora --adapter lora_r16
Metrics: valid_json, exact_match (whole object equal), field_acc (share of gold fields correct)."""
import json, argparse, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
PROMPT = "### Request:\n{text}\n### JSON:\n"
ap = argparse.ArgumentParser()
ap.add_argument("--model", default="Qwen/Qwen2.5-0.5B-Instruct")
ap.add_argument("--mode", choices=["zeroshot", "fewshot", "lora"], required=True)
ap.add_argument("--adapter", default=None)
a = ap.parse_args()

tok = AutoTokenizer.from_pretrained(a.model)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(a.model, torch_dtype=torch.float16).cuda()
if a.mode == "lora":
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, a.adapter)
model.eval()

test = [json.loads(l) for l in open("test.jsonl", encoding="utf-8")]
train = [json.loads(l) for l in open("train.jsonl", encoding="utf-8")]
shots = ""
if a.mode == "fewshot":
    seen_intents, picked = set(), []
    for r in train:
        i = json.loads(r["target"])["intent"]
        if i not in seen_intents: seen_intents.add(i); picked.append(r)
    shots = "".join(PROMPT.format(text=r["text"]) + r["target"] + "\n\n" for r in picked)

def gen(text):
    ids = tok(shots + PROMPT.format(text=text), return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=100, do_sample=False, pad_token_id=tok.pad_token_id)
    s = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return s.split("###")[0].split("\n\n")[0].strip()

def parse(s):
    for cand in (s, s.split("\n")[0]):
        try:
            o = json.loads(cand)
            if isinstance(o, dict): return o
        except Exception: pass
    return None

valid = exact = 0; field = 0.0; preds = []
for r in test:
    pred_s = gen(r["text"]); gold = json.loads(r["target"]); p = parse(pred_s)
    if p is not None:
        valid += 1
        field += sum(p.get(k) == v for k, v in gold.items()) / len(gold)
        exact += p == gold
    preds.append({"text": r["text"], "gold": gold, "pred_raw": pred_s, "heldout_template": r["heldout_template"],
                  "correct": p == gold})
n = len(test)
res = {"mode": a.mode, "n": n, "valid_json": valid / n, "exact_match": exact / n, "field_acc": field / n}
for h in (False, True):
    sub = [x for x in preds if x["heldout_template"] == h]
    res[f"exact_match_{'heldout' if h else 'seen'}_templates"] = sum(x["correct"] for x in sub) / max(1, len(sub))
print(json.dumps(res, indent=2))
tag = a.mode if a.mode != "lora" else f"lora_{a.adapter}"
json.dump({"metrics": res, "predictions": preds}, open(f"results_{tag}.json", "w"), indent=2, ensure_ascii=False)
print("failures to inspect:", [x["text"] for x in preds if not x["correct"]][:5])


## 3. Generate data (1,200 train / 100 test)

In [ ]:
!python gen_data.py

## 4. Baselines: base model zero-shot and 3-shot

In [ ]:
!python -u eval_model.py --mode zeroshot 2>&1 | tee log_eval_zeroshot.txt

In [ ]:
!python -u eval_model.py --mode fewshot 2>&1 | tee log_eval_fewshot.txt

## 5. LoRA fine-tune (r=16, alpha=32, all linear layers, lr 2e-4, 3 epochs) and evaluate

In [ ]:
!python -u train_lora.py --out lora_r16 2>&1 | tee log_train_lora_r16.txt

In [ ]:
!python -u eval_model.py --mode lora --adapter lora_r16 2>&1 | tee log_eval_lora_r16.txt

## 6. QLoRA (4-bit base) - optional, gives a measured memory comparison

In [ ]:
!python -u train_lora.py --qlora --out qlora_r16 2>&1 | tee log_train_qlora_r16.txt

In [ ]:
!python -u eval_model.py --mode lora --adapter qlora_r16 2>&1 | tee log_eval_qlora_r16.txt

## 7. Download everything

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv > gpu_info.txt
!zip -q task1_results.zip results_*.json log_*.txt gpu_info.txt train.jsonl test.jsonl
from google.colab import files; files.download('task1_results.zip')